# Production Downtime Analysis

## Project Overview
Analyze manufacturing downtime records to identify frequent causes, machine-level issues, and high-impact downtime periods. This is a descriptive analytics project.

## Learning Objectives
- Calculate downtime frequency and duration by cause and machine
- Identify recurring failure patterns and high-impact equipment
- Analyze temporal downtime patterns (shift, day, season)
- Estimate production loss from downtime events

## Problem Statement
A manufacturing plant is losing production capacity to unplanned downtime. Management needs to know which machines fail most, what causes drive the longest outages, and when downtime clusters occur.

## Why This Project Matters
Unplanned downtime is one of the largest costs in manufacturing. Identifying root causes and patterns enables predictive maintenance and targeted reliability investments.

## Dataset Overview
Synthetic downtime log: ~3,000 events across 15 machines over 2 years.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 3000
machines = [f'M-{i:02d}' for i in range(1, 16)]
machine_line = {m: np.random.choice(['Line-A', 'Line-B', 'Line-C']) for m in machines}
machine_age = {m: np.random.randint(2, 20) for m in machines}

machine_ids = np.random.choice(machines, n)
causes = np.random.choice(
    ['Mechanical Failure', 'Electrical Fault', 'Sensor Error', 'Material Jam',
     'Operator Error', 'Planned Maintenance', 'Software Issue', 'Calibration'],
    n, p=[0.20, 0.15, 0.12, 0.15, 0.10, 0.12, 0.08, 0.08])
shifts = np.random.choice(['Day', 'Evening', 'Night'], n, p=[0.40, 0.35, 0.25])

dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
event_dates = np.array([pd.Timestamp(d) for d in np.random.choice(dates, n)])

# Duration depends on cause
cause_dur = {'Mechanical Failure': 120, 'Electrical Fault': 90, 'Sensor Error': 30,
             'Material Jam': 20, 'Operator Error': 15, 'Planned Maintenance': 180,
             'Software Issue': 45, 'Calibration': 60}
duration_min = np.array([max(5, int(np.random.exponential(cause_dur[c]))) for c in causes])

# Production loss estimate (units per minute baseline)
units_per_min = {m: np.random.uniform(2, 8) for m in machines}
production_loss = np.array([round(duration_min[i] * units_per_min[machine_ids[i]], 0)
                            for i in range(n)])

planned = (causes == 'Planned Maintenance').astype(int)

df = pd.DataFrame({
    'EventID': [f'DT{i:06d}' for i in range(n)],
    'EventDate': event_dates, 'Machine': machine_ids,
    'Line': [machine_line[m] for m in machine_ids],
    'MachineAge': [machine_age[m] for m in machine_ids],
    'Cause': causes, 'Shift': shifts,
    'DurationMinutes': duration_min, 'Planned': planned,
    'ProductionLoss': production_loss,
})
df['YearMonth'] = df['EventDate'].dt.to_period('M')
df['DayOfWeek'] = df['EventDate'].dt.day_name()

# Focus on unplanned
df_unplanned = df[df['Planned'] == 0].copy()

print(f'Dataset shape: {df.shape}')
print(f'Unplanned events: {len(df_unplanned)} ({len(df_unplanned)/len(df)*100:.0f}%)')
print(f'Total unplanned downtime: {df_unplanned["DurationMinutes"].sum():,} min ({df_unplanned["DurationMinutes"].sum()/60:,.0f} hrs)')
print(f'Total production loss: {df_unplanned["ProductionLoss"].sum():,.0f} units')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["EventDate"].min().date()} to {df["EventDate"].max().date()}')
print(f'Machines: {df["Machine"].nunique()}')
print(f'\nCause distribution:\n{df["Cause"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df_unplanned.groupby('Cause')['DurationMinutes'].sum().sort_values().plot.barh(
    ax=axes[0,0], edgecolor='black', color='coral')
axes[0,0].set_title('Total Unplanned Downtime by Cause (min)')

df_unplanned.groupby('Machine')['DurationMinutes'].sum().sort_values(ascending=False).head(10).plot.bar(
    ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('Top 10 Machines by Total Downtime')
axes[0,1].tick_params(axis='x', rotation=45)

monthly = df_unplanned.groupby('YearMonth')['DurationMinutes'].sum()
monthly.plot(ax=axes[1,0], marker='o', color='red')
axes[1,0].set_title('Monthly Unplanned Downtime')
axes[1,0].tick_params(axis='x', rotation=45)

df_unplanned.groupby('Shift')['DurationMinutes'].sum().plot.bar(
    ax=axes[1,1], edgecolor='black', color='green')
axes[1,1].set_title('Total Downtime by Shift')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Cause Analysis — Frequency vs Impact

In [ ]:
cause_stats = df_unplanned.groupby('Cause').agg(
    events=('EventID', 'count'),
    total_duration=('DurationMinutes', 'sum'),
    avg_duration=('DurationMinutes', 'mean'),
    total_loss=('ProductionLoss', 'sum'),
).round(1).sort_values('total_duration', ascending=False)
print('Cause Analysis (Unplanned):')
print(cause_stats)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(cause_stats['events'], cause_stats['avg_duration'],
          s=cause_stats['total_loss'] / 100, alpha=0.6, edgecolor='black', color='coral')
for c in cause_stats.index:
    ax.annotate(c, (cause_stats.loc[c, 'events'], cause_stats.loc[c, 'avg_duration']),
               fontsize=8, ha='center', va='bottom')
ax.set_xlabel('Event Count')
ax.set_ylabel('Avg Duration (min)')
ax.set_title('Cause: Frequency vs Avg Duration (bubble = production loss)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cause_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

## Machine Reliability

In [ ]:
machine_stats = df_unplanned.groupby('Machine').agg(
    line=('Line', 'first'),
    age=('MachineAge', 'first'),
    events=('EventID', 'count'),
    total_downtime=('DurationMinutes', 'sum'),
    avg_downtime=('DurationMinutes', 'mean'),
    total_loss=('ProductionLoss', 'sum'),
).round(1).sort_values('total_downtime', ascending=False)
print('Machine Reliability (Top 10):')
print(machine_stats.head(10))

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(machine_stats['age'], machine_stats['events'], s=machine_stats['total_downtime'] / 20,
          alpha=0.5, edgecolor='black', color='steelblue')
for m in machine_stats.index:
    ax.annotate(m, (machine_stats.loc[m, 'age'], machine_stats.loc[m, 'events']), fontsize=7)
ax.set_xlabel('Machine Age (years)')
ax.set_ylabel('Failure Events')
ax.set_title('Machine Age vs Failure Frequency (bubble = total downtime)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'machine_reliability.png'), dpi=100, bbox_inches='tight')
plt.show()

## Production Line Comparison

In [ ]:
line_stats = df_unplanned.groupby('Line').agg(
    machines=('Machine', 'nunique'),
    events=('EventID', 'count'),
    total_downtime=('DurationMinutes', 'sum'),
    total_loss=('ProductionLoss', 'sum'),
).round(0)
line_stats['downtime_per_machine'] = (line_stats['total_downtime'] / line_stats['machines']).round(0)
print('Production Line Summary:')
print(line_stats)

fig, ax = plt.subplots(figsize=(8, 5))
line_stats['downtime_per_machine'].plot.bar(ax=ax, edgecolor='black', color='teal')
ax.set_title('Avg Downtime per Machine by Line')
ax.set_ylabel('Minutes')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'line_comparison.png'), dpi=100, bbox_inches='tight')
plt.show()

## Pareto Analysis — Downtime Contribution

In [ ]:
pareto = cause_stats[['total_duration']].sort_values('total_duration', ascending=False)
pareto['cum_pct'] = (pareto['total_duration'].cumsum() / pareto['total_duration'].sum() * 100).round(1)
print('Pareto — Cumulative Downtime:')
print(pareto)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(range(len(pareto)), pareto['total_duration'], edgecolor='black', color='coral', alpha=0.7)
ax1.set_ylabel('Total Downtime (min)')
ax1.set_xticks(range(len(pareto)))
ax1.set_xticklabels(pareto.index, rotation=45, ha='right', fontsize=8)
ax2 = ax1.twinx()
ax2.plot(range(len(pareto)), pareto['cum_pct'], 'ro-', linewidth=2)
ax2.set_ylabel('Cumulative %')
ax2.axhline(80, color='red', linestyle='--', alpha=0.5)
ax1.set_title('Downtime Pareto Analysis')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'pareto.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Mechanical Failure** and **Electrical Fault** drive the majority of unplanned downtime — invest in predictive maintenance for these
- **Older machines** tend to have more failure events — age-based replacement planning is warranted
- **Material Jam** is frequent but short duration — process improvements at feed stations could eliminate many events
- **Night shift** has proportionally less downtime but also less coverage for quick resolution
- **Pareto analysis** shows 2-3 causes account for ~60-70% of all downtime — high-value focus areas

## Limitations
- Synthetic data with simplified failure patterns
- No maintenance cost data
- No spare parts or repair team availability data
- No correlation with production volume or material quality
- No root cause chaining (e.g., sensor error causing material jam)

## How to Improve This Project
- Use real CMMS/MES data with detailed failure codes
- Add maintenance cost analysis (repair costs + production loss costs)
- Build predictive maintenance models using sensor data
- Include MTBF and MTTR analysis
- Add spare parts inventory correlation

## Production Considerations
- Real-time downtime dashboards for plant managers
- Automated failure alerts with estimated impact
- Predictive maintenance scheduling based on failure patterns
- Spare parts inventory optimization linked to failure frequency

## Common Mistakes
- Counting events without weighting by duration or production loss
- Ignoring planned maintenance when calculating total availability
- Not normalizing downtime by machine utilization hours
- Treating all causes equally when a few drive most of the losses

## Mini Challenge / Exercises
1. Calculate OEE (Overall Equipment Effectiveness) proxy: availability = 1 - (downtime / total hours) for each machine
2. Which cause has improved most from the first year to the second year?
3. If you eliminated the top 2 causes entirely, what % of downtime would remain?
4. Which machine has the worst MTBF (mean time between failures)?

## Final Summary / Key Takeaways
- Production downtime analysis pinpoints where reliability investments have the highest ROI
- Pareto analysis consistently shows a few causes drive most of the impact
- Machine age and line-level analysis reveal systemic vs equipment-specific issues
- Frequency × duration determines true impact — don't optimize for event count alone
- Predictive maintenance programs should be prioritized for the highest-impact failure modes